In [ ]:
# this code enables us to view multiple outputs at same time 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
#load dataset
taxi = pd.read_csv("yellow_tripdata.csv")

#inspecting data
taxi.shape
taxi.head()
taxi.tail()
taxi.info()
taxi.isna().sum()
taxi.describe()


In [ ]:

taxi['tip_amount'].describe()
sns.histplot(data=taxi, x="tip_amount", bins = 30,
             color="blue")
plt.title("Distribution of tip_amount($)")
plt.show()

#doing further analysis to ubderstand the dsitatibution
Q1 = taxi['tip_amount'].quantile(0.25)
Q3 = taxi['tip_amount'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = taxi[(taxi['tip_amount'] < lower_bound) | (taxi['tip_amount'] > upper_bound)]


print("Lower bound: " ,lower_bound)
print("Upper bound: " ,upper_bound)
print("Number of outliers: ", len(outliers))
print("Percentage of data: ", (len(outliers)/len(taxi)) * 100,"%")

## Taxi Data Exploration: What is the shape of the data? Are there missing values or impossible values (e.g. negative tips, zero-distance trips)? What does the tip_amount distribution look like (skew, outliers, many zeros)? How will this influence your preprocessing?

The dataset has a shape of **41,202 rows and 13 columns**. Upon inspection, there are no null or missing values in any column. Using the describe() function, we obtained a 5-number statistical summary for each column.

The tip_amount histogram shows a **right-skewed distribution** ie tthe bulk of trips have tip amounts concentrated between **$10–$20**, with a long tail stretching toward $98, indicating a small number of unusually high tips.

Performing the IQR-based outlier calc, **Lower bound = $0.715, Upper bound = $25.475**

Since the minimum tip is **$0.01** and the maximum is **$98.00**, both bounds are violated, confirming the presence of outliers.There is total of **386 rows (0.94%) of the data** were identified as outliers. Since this represents less than 1% of the data, removing them will not significantly reduce the dataset size, but will prevent extreme values from biasing the regression model during training.

In [ ]:

# TODO 1: Handle missing / invalid rows

taxi_cleaned = taxi[(taxi['tip_amount'] >= lower_bound) & (taxi['tip_amount'] <= upper_bound)]   # Remove tip_amount outliers using IQR bounds calculated earlier
taxi_cleaned = taxi_cleaned[taxi_cleaned['trip_distance'] > 0]
taxi_cleaned = taxi_cleaned[taxi_cleaned['fare_amount'] > 0]
taxi_cleaned = taxi_cleaned[taxi_cleaned['tip_amount'] > 0]

# assumed data recording errors in trip_distance
taxi_cleaned = taxi_cleaned[taxi_cleaned['trip_distance'] >= 0.10]   # under 160m unrealistic
taxi_cleaned = taxi_cleaned[taxi_cleaned['trip_distance'] <= 50]     # over 50 miles unrealistic

print(f"Rows before cleaning: {len(taxi)}")
print(f"Rows after cleaning: {len(taxi_cleaned)}")
print(f"Rows removed: {len(taxi) - len(taxi_cleaned)} ({(len(taxi) - len(taxi_cleaned))/len(taxi)*100:.2f}%)")


# TODO 2: Feature engineering


# Feature 1: fare per mile 
taxi_cleaned['fare_per_mile'] = taxi_cleaned['fare_amount'] / taxi_cleaned['trip_distance']

# Feature 2: total surcharges which combines all extra charges
taxi_cleaned['total_surcharges'] = (taxi_cleaned['mta_tax'] + 
                                     taxi_cleaned['tolls_amount'] + 
                                     taxi_cleaned['improvement_surcharge'])

# Feature 3: total cost which is full trip cost excluding tip
taxi_cleaned['total_cost'] = taxi_cleaned['fare_amount'] + taxi_cleaned['total_surcharges']

print("New features:")
print(taxi_cleaned[['fare_per_mile', 'total_surcharges', 'total_cost']].describe())


# TODO 3: Decide categorical vs numeric and encode


# Drop constant, nearly constant and redundant columns which do ot help for prediction ie zero predictive value

taxi_cleaned = taxi_cleaned.drop(columns=[
    'VendorID',                                           # constant ;all values are 2
    'payment_type',                                       # constant : all values are 1
    'improvement_surcharge',                              # used in feature engineering, now redundant
    'store_and_fwd_flag',                                 # nearly constant , standard dev of 0.02
    'PULocationID',                                       # nearly constant , std dev of 7.44
    'mta_tax',                                            # used in feature engineering, now redundant
    'tolls_amount',                                       # absorbed into total_surcharges, now redundant
    'RatecodeID',                                         #  nearly constant, most value is 2
])

print("Remaining columns:", taxi_cleaned.columns.tolist())
print("Shape:", taxi_cleaned.shape)


# TODO 4: Define X and y then scale 
X = taxi_cleaned.drop(columns=['tip_amount'])  #features
y = taxi_cleaned['tip_amount']     #target



Rows before cleaning: 41202
Rows after cleaning: 40718
Rows removed: 484 (1.17%)
New features:
       fare_per_mile  total_surcharges    total_cost
count   40718.000000      40718.000000  40718.000000
mean        4.026369          7.097066     77.007974
std         8.633205          2.958903      6.679634
min         1.972387          1.000000      6.600000
25%         3.625065          8.440000     78.440000
50%         3.858875          8.440000     78.440000
75%         4.036909          8.440000     78.440000
max       653.846154         41.640000    180.200000
Remaining columns: ['passenger_count', 'trip_distance', 'DOLocationID', 'fare_amount', 'tip_amount', 'fare_per_mile', 'total_surcharges', 'total_cost']
Shape: (40718, 8)


## Student Reasoning — Taxi Preprocessing

**1. How did you handle missing/invalid rows and why?**

Upon initial inspection using .isna().sum(), the dataset contained no null or 
missing values in any column. However, several invalid rows were identified and removed:

-**tip_amount outliers** :  using IQR-based outlier detection, I calculated a 
lower bound of **$0.715** and an upper bound of **$25.475**. Any tip outside 
these bounds was removed, affecting **386 rows (0.94%)**. I removed these because 
extreme tip values would bias the regression model toward unrealistic predictions.

**trip_distance ≤ 0 and fare_amount ≤ 0** : rows with zero or negative distances 
and fares are physically impossible and were removed immediately.

 **trip_distance < 0.10 miles** : when I created the fare_per_mile feature, I 
discovered rows with distances as small as 0.01 miles (approx 16 meters) producing impossible 
fare_per_mile values like $8,500. I assumed these to be data recording errors or GPS errors, as a 
trip of 16 meters is not realistic under any circumstance. I removed all 23 rows below 
0.10 miles (160 meters) because keeping them would distort the fare_per_mile feature, 
causing the regression model to learn incorrect relationships and produce unreliable 
predictions.

 **trip_distance > 50 miles** : only 2 rows exceeded 50 miles, with distances up to 
189 miles. I assumed these to be data recording errors as such distances are 
unrealistically long for a city taxi trip. I chose 50 miles as a reasonable upper 
threshold based on the assumption that no legitimate city taxi trip should exceed 
this distance.

In total, **484 rows (1.17%)** were removed, retaining **98.83%** of the original 
41,202 rows which is approx a negligible loss that results in a clean, reliable dataset for modelling.



**2. Which new feature(s) did you engineer and what is the intuition behind them?**

I engineered three new features, all carefully designed to avoid target leakage 
no feature uses tip_amount directly since that is the variable I am trying to predict.

**fare_per_mile**:  I created this feature because i assumed that the cost per mile of a trip is a stronger signal for tipping behaviour than fare or distance alone. A passenger on a more expensive per-mile trip may be more 
likely to tip generously than one on a cheaper route.

**total_surcharges**:  I combined these three separate charge columns into one feature 
because I assumed that the total additional cost burden on a passenger influences their tipping decision. A passenger 
already paying high surcharges is on a more expensive trip overall, which I assumed correlates with higher tips.

**total_cost** : I created this feature because i assumed that the total amount a passenger pays,
 excluding the tip, is a good indicator of how much they might tip. A passenger paying a higher total is likely on a longer or more premium trip and may tip more generously.

After creating these features, the original component columns (mta_tax, tolls_amount, 
improvement_surcharge) were dropped as they were now redundant, their information is 
fully captured in the engineered features.


**3 Which scaling method did you choose and why is it appropriate here?**

I chose StandardScaler* from sklearn.preprocessing, which scales each feature 
to have a mean of 0 and a standard deviation of 1. This is appropriate here because:

The numeric features have very different scales :for example, fare_amount ranges 
from $5 to $178 while passenger_count ranges from 1 to 8. Without scaling, features 
with larger ranges would dominate the model and those with smaller ranges would be ignored.
